# 🔬 Model Pipeline: BioMistral-7B Biomedical LLM (4-bit QLoRA)
### **Jadavpur University Research Internship | Under: Prof. Tohida Rehman**
**Track:** FIRE MixSub-SciHigh Scientific Highlight Summarization
**Model ID:** `BioMistral/BioMistral-7B`
**Architecture:** CAUSAL (4-bit QLoRA)

---
### 🎯 Strict Uniform Benchmark Constraints (Scientifically Fair)
- **`MAX_SOURCE_LENGTH = 512` tokens**: Uniform input length across all 6 models.
- **`MAX_TARGET_LENGTH = 128` tokens**: Uniform output generation limit across all 6 models.

### 📊 The 3 Non-Redundant Benchmark Pillars Measured:
1. **Overall Quality**: ROUGE-L F1 (↑ Higher is Better) & Target F1 Score (↑)
2. **Hallucination Risk**: Ungrounded Token % (↓ Lower is Better) & Precision Source (% grounded in abstract)
3. **Inference Time**: Wall-Clock Latency per Sample (↓ Lower is Faster) & Throughput (tok/s ↑)

In [ ]:

# CELL 1: Google Drive and project configuration

from google.colab import drive
import os

drive.mount("/content/drive")

DRIVE_BASE_DIR = "/content/drive/MyDrive/JU_Benchmark_Project"

MODEL_CHECKPOINT_DIR = os.path.join(
    DRIVE_BASE_DIR, "checkpoints", "biomistral-7b"
)

RESULTS_DIR = os.path.join(DRIVE_BASE_DIR, "results")
DATASET_DIR = os.path.join(DRIVE_BASE_DIR, "Dataset")

os.makedirs(MODEL_CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DATASET_DIR, exist_ok=True)

MODEL_ID = "BioMistral/BioMistral-7B"

MAX_SOURCE_LENGTH = 512
MAX_TARGET_LENGTH = 128
MAX_SEQUENCE_LENGTH = MAX_SOURCE_LENGTH + MAX_TARGET_LENGTH

print("Google Drive mounted successfully.")
print("Model:", MODEL_ID)
print("Checkpoint directory:", MODEL_CHECKPOINT_DIR)
print("Results directory:", RESULTS_DIR)
print("Dataset directory:", DATASET_DIR)
print("Input limit:", MAX_SOURCE_LENGTH)
print("Output limit:", MAX_TARGET_LENGTH)


In [ ]:

# CELL 2: Install libraries and verify GPU

%pip install -q transformers accelerate bitsandbytes peft datasets rouge-score sentencepiece huggingface_hub

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU not detected. Select Runtime > Change runtime type > T4 GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))

gpu_memory = (
    torch.cuda.get_device_properties(0).total_memory / 1e9
)

print(f"GPU memory: {gpu_memory:.2f} GB")

USE_BF16 = torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = (
    torch.bfloat16 if USE_BF16 else torch.float16
)

print("Compute dtype:", COMPUTE_DTYPE)


In [ ]:

# CELL 3: Load FIRE MixSub-SciHigh datasets

import pandas as pd

TRAIN_CSV = os.path.join(
    DATASET_DIR, "MixSub-SciHigh_train_FIRE.csv"
)

VAL_CSV = os.path.join(
    DATASET_DIR, "MixSub-SciHigh_val_FIRE.csv"
)

TEST_CSV = os.path.join(
    DATASET_DIR, "MixSub-SciHigh_test_FIRE(Masked).csv"
)

# Check required training and validation files
for name, path in [
    ("Training", TRAIN_CSV),
    ("Validation", VAL_CSV)
]:
    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"{name} dataset not found:\n{path}\n"
            "Upload the CSV into the Dataset folder in Google Drive."
        )

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)

# Validate required columns
required_columns = {"Abstract", "Highlights"}

for name, df in [
    ("Training", train_df),
    ("Validation", val_df)
]:
    missing = required_columns - set(df.columns)

    if missing:
        raise ValueError(
            f"{name} CSV is missing columns: {missing}"
        )

# Clean training and validation data
def clean_dataset(df):
    df = df.dropna(
        subset=["Abstract", "Highlights"]
    ).copy()

    df["Abstract"] = df["Abstract"].astype(str).str.strip()
    df["Highlights"] = df["Highlights"].astype(str).str.strip()

    df = df[
        (df["Abstract"] != "") &
        (df["Highlights"] != "")
    ]

    return df.reset_index(drop=True)


train_df = clean_dataset(train_df)
val_df = clean_dataset(val_df)

# Test file is optional
if os.path.isfile(TEST_CSV):
    test_df = pd.read_csv(TEST_CSV)

    if "Abstract" not in test_df.columns:
        raise ValueError(
            "Masked test CSV must contain an Abstract column."
        )

    test_df["Abstract"] = (
        test_df["Abstract"].fillna("").astype(str)
    )

    print("Test samples:", len(test_df))
else:
    test_df = None
    print("Masked test CSV not found. Test prediction will be skipped.")

print("\nTraining samples:", len(train_df))
print("Validation samples:", len(val_df))

print("\nExample abstract:")
print(train_df.iloc[0]["Abstract"][:300])

print("\nExample highlight:")
print(train_df.iloc[0]["Highlights"][:300])


In [ ]:

# CELL 4: Three-pillar benchmark metrics

import re
import numpy as np

from collections import Counter
from rouge_score import rouge_scorer


def clean_tokenize(text):
    if not isinstance(text, str):
        return []

    return re.findall(
        r"\b[a-zA-Z0-9]+(?:'[a-zA-Z0-9]+)?\b",
        text.lower()
    )


def get_ngrams(tokens, n=1):
    if len(tokens) < n or n < 1:
        return []

    return [
        tuple(tokens[i:i+n])
        for i in range(len(tokens) - n + 1)
    ]


def compute_pillars_for_sample(
    gen_text,
    src_text,
    tgt_text
):
    gen_tokens = clean_tokenize(gen_text)
    src_tokens = clean_tokenize(src_text)
    tgt_tokens = clean_tokenize(tgt_text)

    gen_counts = Counter(get_ngrams(gen_tokens, 1))
    src_counts = Counter(get_ngrams(src_tokens, 1))
    tgt_counts = Counter(get_ngrams(tgt_tokens, 1))

    total_gen = sum(gen_counts.values())
    total_tgt = sum(tgt_counts.values())

    # Pillar 2: Source grounding proxy
    overlap_src = sum(
        (gen_counts & src_counts).values()
    )

    precision_src = (
        overlap_src / total_gen
        if total_gen > 0 else 0.0
    )

    hallucination_risk = 1.0 - precision_src

    # Pillar 1: Target highlight alignment
    overlap_tgt = sum(
        (gen_counts & tgt_counts).values()
    )

    precision_tgt = (
        overlap_tgt / total_gen
        if total_gen > 0 else 0.0
    )

    recall_tgt = (
        overlap_tgt / total_tgt
        if total_tgt > 0 else 0.0
    )

    f1_tgt = (
        2 * precision_tgt * recall_tgt /
        (precision_tgt + recall_tgt)
        if (precision_tgt + recall_tgt) > 0
        else 0.0
    )

    return {
        "hallucination_risk": hallucination_risk,
        "precision_src": precision_src,
        "precision_tgt": precision_tgt,
        "recall_tgt": recall_tgt,
        "f1_tgt": f1_tgt
    }


def evaluate_core_benchmarks(
    predictions,
    sources,
    references,
    latencies=None,
    generated_tokens=None
):
    if not (
        len(predictions) == len(sources) == len(references)
    ):
        raise ValueError(
            "Predictions, sources and references must have equal lengths."
        )

    if len(predictions) == 0:
        raise ValueError("No samples available for evaluation.")

    scorer = rouge_scorer.RougeScorer(
        ["rougeL"],
        use_stemmer=True
    )

    rouge_f1 = []
    rouge_precision = []
    rouge_recall = []

    for pred, ref in zip(predictions, references):
        score = scorer.score(ref, pred)

        rouge_f1.append(score["rougeL"].fmeasure)
        rouge_precision.append(score["rougeL"].precision)
        rouge_recall.append(score["rougeL"].recall)

    sample_metrics = [
        compute_pillars_for_sample(pred, src, ref)
        for pred, src, ref in zip(
            predictions, sources, references
        )
    ]

    results = {
        # Pillar 1: Quality
        "overall_quality_rougeL_f1": float(
            np.mean(rouge_f1)
        ),
        "rougeL_precision": float(
            np.mean(rouge_precision)
        ),
        "rougeL_recall": float(
            np.mean(rouge_recall)
        ),
        "f1_score_target": float(
            np.mean([x["f1_tgt"] for x in sample_metrics])
        ),
        "precision_target": float(
            np.mean([x["precision_tgt"] for x in sample_metrics])
        ),
        "recall_target": float(
            np.mean([x["recall_tgt"] for x in sample_metrics])
        ),

        # Pillar 2: Grounding proxy
        "hallucination_risk_pct": float(
            np.mean([
                x["hallucination_risk"]
                for x in sample_metrics
            ])
        ),
        "source_grounding_precision": float(
            np.mean([
                x["precision_src"]
                for x in sample_metrics
            ])
        )
    }

    # Pillar 3: Inference speed
    if latencies is not None and len(latencies) > 0:
        results["wall_clock_latency_sec"] = float(
            np.mean(latencies)
        )

        results["total_wall_clock_sec"] = float(
            np.sum(latencies)
        )

    if (
        generated_tokens is not None
        and latencies is not None
        and len(latencies) > 0
    ):
        total_tokens = sum(generated_tokens)
        total_time = sum(latencies)

        results["throughput_tokens_per_sec"] = float(
            total_tokens / max(total_time, 1e-8)
        )

    return results


print("Three-pillar benchmark engine loaded successfully.")



In [ ]:

# FIXED CELL: TrainingArguments + Trainer initialization
# Compatible with different Transformers versions

import inspect
import torch
from transformers import TrainingArguments, Trainer

# 1. Detect supported TrainingArguments parameters
supported_params = inspect.signature(
    TrainingArguments.__init__
).parameters

# 2. Define training configuration
training_kwargs = {
    "output_dir": MODEL_CHECKPOINT_DIR,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "learning_rate": 2e-4,
    "num_train_epochs": 3,
    "logging_steps": 10,
    "save_steps": 100,
    "save_total_limit": 2,
    "fp16": not USE_BF16,
    "bf16": USE_BF16,
    "gradient_checkpointing": True,
    "report_to": "none",
    "remove_unused_columns": False,
}

# 3. Add optional parameters only if supported
optional_kwargs = {
    "save_strategy": "steps",
    "warmup_ratio": 0.03,
    "lr_scheduler_type": "cosine",
    "optim": "paged_adamw_8bit",
    "gradient_checkpointing_kwargs": {
        "use_reentrant": False
    },
}

for key, value in optional_kwargs.items():
    if key in supported_params:
        training_kwargs[key] = value

# 4. Remove every unsupported argument
training_kwargs = {
    key: value
    for key, value in training_kwargs.items()
    if key in supported_params
}

# 5. Initialize training arguments
training_args = TrainingArguments(**training_kwargs)

print("TrainingArguments initialized successfully!")
print("Transformers version:",
      __import__("transformers").__version__)
print("Warmup ratio enabled:",
      "warmup_ratio" in training_kwargs)

# 6. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    data_collator=data_collator
)

print("Trainer initialized successfully!")
print("Ready to start BioMistral-7B QLoRA training.")



In [ ]:

# FAST: BioMistral zero-shot benchmark (no training required)

import os, json, time, torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "BioMistral/BioMistral-7B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map="auto"
)
model.eval()

df = pd.read_csv(VAL_CSV).dropna(
    subset=["Abstract", "Highlights"]
).head(20)

predictions, latencies, token_counts = [], [], []

for abstract in df["Abstract"]:
    prompt = (
        "[INST] Extract concise scientific highlights "
        "from this abstract:\n" + str(abstract) + " [/INST]"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda")

    start = time.perf_counter()

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False
        )

    torch.cuda.synchronize()
    latencies.append(time.perf_counter() - start)

    generated = output[0][inputs["input_ids"].shape[1]:]

    predictions.append(
        tokenizer.decode(generated, skip_special_tokens=True).strip()
    )
    token_counts.append(len(generated))

df["Generated_Highlights"] = predictions
df.to_csv(
    os.path.join(RESULTS_DIR, "biomistral_zero_shot_predictions.csv"),
    index=False
)

metrics = evaluate_core_benchmarks(
    predictions,
    df["Abstract"].tolist(),
    df["Highlights"].tolist(),
    latencies,
    token_counts
)

metrics.update({
    "model": MODEL_ID,
    "fine_tuning": False,
    "evaluation_samples": len(df),
    "evaluation_type": "Zero-shot"
})

with open(
    os.path.join(RESULTS_DIR, "benchmark_biomistral-7b.json"),
    "w"
) as f:
    json.dump(metrics, f, indent=4)

print("ZERO-SHOT BENCHMARK COMPLETED")
print(json.dumps(metrics, indent=4))


In [ ]:

# Save and verify BioMistral results in Google Drive

import os
import json
import shutil

from google.colab import drive

# Ensure Google Drive is mounted
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

RESULTS_DIR = "/content/drive/MyDrive/JU_Benchmark_Project/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# Save benchmark metrics
benchmark_path = os.path.join(
    RESULTS_DIR, "benchmark_biomistral-7b.json"
)

with open(benchmark_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=4, ensure_ascii=False)

# Save predictions
prediction_path = os.path.join(
    RESULTS_DIR, "biomistral_zero_shot_predictions.csv"
)

df.to_csv(prediction_path, index=False)

# Verify saved files
print("BioMistral results saved successfully!")

for filename in [
    "benchmark_biomistral-7b.json",
    "biomistral_zero_shot_predictions.csv"
]:
    path = os.path.join(RESULTS_DIR, filename)
    print(
        filename,
        "| Exists:", os.path.isfile(path),
        "| Size:", os.path.getsize(path), "bytes"
    )

print("\nGoogle Drive folder:", RESULTS_DIR)
